## 환경 변수 파일 생성

In [1]:
import os
import pathlib

here = pathlib.Path.cwd()
ROOT = here.parents[2] if here.name == "day01" else here
print(ROOT)

os.chdir(ROOT)  # 모든 상대경로는 이 폴더가 기준이 된다.

c:\hw\hw\python_basic\hanhwa-agent


In [2]:
# .env 파일에 적힌 내용을 현재 파일로 가져오기
from dotenv import load_dotenv
import os

# .env를 찾아 읽고, os.environ에 채워넣는다.
# override=True : 이미 있는 환경변수도 .env 값으로 덮어쓴다.
# load_dotenv(override=True)
load_dotenv()
print("APP_MODE:", os.getenv("APP_MODE"))
print("ANTHROPIC_API_KEY:", os.environ.get("ANTHROPIC_API_KEY"))

APP_MODE: mock
ANTHROPIC_API_KEY: claude-api-key


### - Pydantic을 활용하여 env 정보 가져오는 방식

In [3]:
# pydantic-settings 설치 필요
from pydantic_settings import BaseSettings, SettingsConfigDict
from pydantic import Field, SecretStr
from functools import lru_cache

class Settings(BaseSettings):
    # 앱 설정: 환경변수와 .env에서 값을 읽어오는 처리
    model_config = SettingsConfigDict(
        env_file=".env",
        env_file_encoding="utf-8",
        extra="ignore",             # .env에 모르는 항목이 있어도 무시
    )

    # 필드: 필드명이 곧 환경변수 이름이다. (대소문자 구분 x)
    app_mode: str = "dev"
    anthropic_api_key: SecretStr        # print나 log에 값이 안찍힘
    llm_model: str = "claude-haiku-4-5"

    # 타입이 int이므로 "400"을 자동으로 형변환
    max_tokens: int = Field(default=400, ge=1, le=8192)
    daily_call_limit: int = Field(default=200, ge=1)
    max_input_chars: int = Field(default=200, ge=1)

# settings = Settings()
# print("app_mode:", settings.app_mode)
# print("anthropic_api_key:", settings.anthropic_api_key)
# print("max_tokens:", settings.max_tokens, type(settings.max_tokens))

@lru_cache      # 이 함수는 처음 한번만 실행, 두번째부터는 캐시된 데이터 반환
def get_settings() -> Settings:
    print("Settings 객체 생성")
    return Settings()

print("첫번쨰 호출")
s1 = get_settings()
print("두번째 호출")
s2 = get_settings()
print("같은 객체??:", s1 is s2)

첫번쨰 호출
Settings 객체 생성
두번째 호출
같은 객체??: True
